# [TITLE: something catchy about the "can't lose" betting strategy]

*Your name · Computational Probability · date*

## 1. Hook and the big question

<!-- TODO: 2-3 sentences that grab a freshman. A meme, a casino anecdote, or "what if I told you there's a betting strategy that always ends in profit?" -->

**Is the Martingale betting strategy actually profitable?**

## 2. Roulette and the Martingale strategy

<!-- TODO: explain in your own words, for someone who has never seen roulette.
Cover: 38 pockets (18 red, 18 black, 2 green), a $1 bet on black/red pays $1 profit,
so P(win) = 18/38. Then explain Martingale: bet $1, double after each loss, reset to $1 after a win. -->

<!-- TODO: walk through ONE short example sequence (e.g. Black, Black, Red) in a small table,
showing why it *seems* to always end at +$1. -->

## 3. Why simulate?

<!-- TODO: 2-4 sentences. Idea: instead of working out the math, we let the computer play the
game thousands of times and look at what happens on average. -->

## 4. The rules of our simulated casino

<!-- TODO: explain each parameter in plain language and why it matters. -->

| Parameter | Meaning | Starting value |
|-----------|---------|----------------|
| **B** | Starting budget | $200 |
| **W** | Stop when we reach this much money | $300 |
| **L** | Stop after this many spins | 200 |
| **M** | Casino's maximum bet | $100 |

A visit ends when the player (1) reaches **W**, (2) goes bankrupt, or (3) completes **L** spins.

## 5. Building the simulation

<!-- TODO: before the code, say in plain language what the function is going to do.
After the code, explain the pieces: the bet, the spin, the stopping check. -->

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P_WIN = 18 / 38  # chance a bet on black (or red) wins

def play_visit(B=200, W=300, L=200, M=100, rng=None):
    """Simulate one casino visit using the Martingale strategy.
    Returns the list of budgets: the starting budget, then the budget after each spin."""
    if rng is None:
        rng = np.random.default_rng()
    budget = [B]
    next_bet = 1
    for _ in range(L):
        bet = min(next_bet, M, budget[-1])      # can't bet more than the max or more than we have
        won = rng.random() < P_WIN              # spin the wheel
        budget.append(budget[-1] + bet if won else budget[-1] - bet)
        if budget[-1] >= W or budget[-1] <= 0:  # hit the goal, or bankrupt
            break
        next_bet = 1 if won else 2 * bet        # Martingale: reset after a win, double after a loss
    return budget

## 6. A few visits to the casino

<!-- TODO: after the plot, describe what you see. How many went bankrupt? How many hit the goal?
How many ran out of spins? -->

In [ ]:
rng = np.random.default_rng(12345)  # TODO: pick your own seed

plt.figure(figsize=(8, 5))
for _ in range(5):
    plt.plot(play_visit(rng=rng))
plt.axhline(300, linestyle="--", color="gray")  # goal W
plt.axhline(0, linestyle="--", color="gray")    # bankruptcy
plt.xlabel("Spin number")
plt.ylabel("Budget ($)")
plt.title("Five simulated casino visits")
plt.show()

## 7. Average profit over thousands of visits

<!-- TODO: explain why one visit isn't enough and why averaging over many visits helps.
Then interpret the number below. -->

In [ ]:
R = 10_000
final_budgets = [play_visit(rng=rng)[-1] for _ in range(R)]
profits = np.array(final_budgets) - 200

print(f"Average profit over {R:,} visits: ${profits.mean():.2f}")
print(f"Share of visits that ended with a profit: {(profits > 0).mean():.1%}")
print(f"Share of visits that ended in bankruptcy: {(np.array(final_budgets) <= 0).mean():.1%}")

## 8. Does changing the rules change the answer?

<!-- TODO: say which parameter you're changing and why you picked it.
Sketch: M (the casino's max bet) is changed below, but you can swap in B, W, or L. -->

In [ ]:
max_bets = [10, 25, 50, 100, 200, 400]
R = 10_000

avg_profit = []
for m in max_bets:
    finals = [play_visit(M=m, rng=rng)[-1] for _ in range(R)]
    avg_profit.append(np.mean(finals) - 200)

plt.figure(figsize=(8, 5))
plt.plot(max_bets, avg_profit, marker="o")
plt.axhline(0, linestyle="--", color="gray")
plt.xlabel("Casino's maximum bet, M ($)")
plt.ylabel("Average profit ($)")
plt.title("Average profit vs. maximum bet")
plt.show()

<!-- TODO: interpret the plot. Does the parameter matter? Is profit ever positive? -->

## 9. Take-home message

<!-- TODO: directly answer "Is the strategy actually profitable?"
Explain WHY it looks safe (lots of small wins) and why it isn't (rare, huge losses
that wipe out many small wins). End with something memorable. -->